In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!cp /content/drive/MyDrive/modules_work/COMP34612/comp34612.zip /content/

<a href="https://colab.research.google.com/github/lpankhurst/t2-cwk/blob/main/comp34612_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:

- change instances of dP/dD to dP/dL

# Install and Import

In [3]:
import zipfile, os, random, gc, math
from IPython.display import Javascript
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 15)
pd.options.display.float_format = '{:.6f}'.format

In [4]:
extract_path = "."
zip_filename = "comp34612.zip"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [5]:
!pip install xlsxwriter

In [6]:
##  Note that xlsxwriter is essential for these imports to work (and therefore the simulation entirely).

from engine import Engine
from gui import GUI

Please import the necessary packages here or use `!pip install *package_name*` if they are not already installed.

# Base Leader and Example Leader

## Base Leader

This is the base class for all leader subclasses.

In [7]:
class Leader:
    _subclass_registry = {}

    def __init__(self, name, engine):
        self.name = name
        self.engine = engine

    @classmethod
    def cleanup_old_subclasses(cls):
        """
        A function to remove old subclasses before defining new ones.
        """
        existing_subclasses = list(cls.__subclasses__())

        for subclass in existing_subclasses:
            subclass_name = subclass.__name__
            if subclass_name in cls._subclass_registry:
                del cls._subclass_registry[subclass_name]
                del subclass
        gc.collect()

    @classmethod
    def update_subclass_registry(cls):
        """
        A function to update registry after cleaning up old subclasses.
        """
        cls.cleanup_old_subclasses()
        cls._subclass_registry = {subclass.__name__: subclass for subclass in cls.__subclasses__()}

    def new_price(self, date):
        """
        A function for setting the new price of each day.
        :param date: date of the day to be updated
        :return: (float) price for the day
        """
        pass

    def get_price_from_date(self, date):
        """
        A function for getting the price set on a date.
        :param date: (int) date to get the price from
        :return: a tuple (leader_price, follower_price)
        """
        return self.engine.exposed_get_price(date)


    def start_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

    def end_simulation(self):
        """
        A function runs at the beginning of the simulation.
        """
        pass

## Example Leader

This is a simple example leader subclass.

In [8]:
class SimpleLeader(Leader):
    def __init__(self, name, engine):
        super().__init__(name, engine)

    def new_price(self, date: int):
        return 1.5 + random.random() * 0.1

## Generic functions

In [9]:
def load_xls_to_df() -> dict[str, pd.DataFrame]:

  ##  Converts Excel file to dictionary of Pandas dataframes.

  dfs = pd.read_excel('data.xlsx', sheet_name = None, index_col = "Date")

  return dfs

In [10]:
def loaded_dfs_setup(dfs: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:

  ##  Sets up extra dataframe columns; namely profit and n-order derivatives.

  ##  S = 100 − 5uL + 3uF
  ##  P = (u_L - u_C) * S

  dfs_new = {}

  for follower, df_follower in dfs.items():

    dfs_new[follower] = df_follower.copy()

    dfs_new[follower]["Leader's Price dL/dD"] = dfs_new[follower]["Leader's Price"].diff().fillna(0)

    dfs_new[follower]["Leader's Price d2L/dD2"] = dfs_new[follower]["Leader's Price dL/dD"].diff().fillna(0)

    dfs_new[follower]["Profit"] = (df_follower["Leader's Price"] - df_follower["Cost"]) * (100 - 5 * df_follower["Leader's Price"] + 3 * df_follower["Follower's Price"])

    dfs_new[follower]["Profit dP/dD"] = dfs_new[follower]["Profit"].diff().fillna(0)

    dfs_new[follower]["Profit d2P/dD2"] = dfs_new[follower]["Profit dP/dD"].diff().fillna(0)

    dfs_new[follower]["Profit d3P/dD3"] = dfs_new[follower]["Profit d2P/dD2"].diff().fillna(0)

  return dfs_new

In [11]:
dfs_loaded = load_xls_to_df()

##  print(dfs_loaded)

dfs_primed = loaded_dfs_setup(dfs_loaded)

print(dfs_primed["Follower_Mk3"].head(2))

      Leader's Price  Follower's Price  Cost  Leader's Price dL/dD  \
Date                                                                 
1           1.780782          1.278121     1              0.000000   
2           1.730777          1.437587     1             -0.050005   

      Leader's Price d2L/dD2    Profit  Profit dP/dD  Profit d2P/dD2  \
Date                                                                   
1                   0.000000 74.120001      0.000000        0.000000   
2                  -0.050005 69.905300     -4.214702       -4.214702   

      Profit d3P/dD3  
Date                  
1           0.000000  
2          -4.214702  


In [12]:
def extract_initial_stats(df: pd.DataFrame,
                          initial_boost: int) -> dict[str, float]:

  ##  Derives data to write to date 101.

  stats = {}

  res = df.sort_values(by = "Profit").iloc[74] ##.iloc[0]

  print(df["Leader's Price"].quantile(0.75))

  # stats["Leader's Price"] = res["Leader's Price"]

  stats["Leader's Price"] = res["Cost"]

  stats["Follower's Price"] = res["Leader's Price"]

  stats["Cost"] = res["Cost"]

  stats["Leader's Price dL/dD"] = abs(res["Leader's Price dL/dD"]) * initial_boost

  stats["Leader's Price d2L/dD2"] = 0

  stats["Profit"] = 0

  stats["Profit dP/dD"] = 0

  stats["Profit d2P/dD2"] = 0

  stats["Profit d3P/dD3"] = 0

  stats_floats = {}

  for k, v in stats.items():

    stats_floats[k] = float(v)

  return stats_floats

In [13]:
def update_history(df: pd.DataFrame,
                   date: int,
                   cost: float,
                   prices: tuple[float, float],
                   row_stats: dict[str, float]) -> pd.DataFrame:

  ##  Updates 30-day dataframe with full row of data, based on latest leaders & follower prices.

  print("update_history()")

  new_row = {}

  new_row["Cost"] = cost

  new_row["Leader's Price"] = prices[0]

  new_row["Follower's Price"] = prices[1]

  print(cost, prices[0], prices[1])

  new_row["Profit"] = (new_row["Leader's Price"] - new_row["Cost"]) * (100 - 5 * new_row["Leader's Price"] + 3 * new_row["Follower's Price"])

  print(f"new row in progress for date {date}", new_row)

  if (date == 101):

    for field in ["Leader's Price dL/dD",
                  "Leader's Price d2L/dD2",
                  "Profit dP/dD",
                  "Profit d2P/dD2",
                  "Profit d3P/dD3"]:

      new_row[field] = row_stats[field]

  else:

    new_row["Leader's Price dL/dD"] = new_row["Leader's Price"] - df.loc[date - 1]["Leader's Price"]

    new_row["Leader's Price d2L/dD2"] = new_row["Leader's Price dL/dD"] - df.loc[date - 1]["Leader's Price dL/dD"]

    print("new row LP done", new_row)

    new_row["Profit dP/dD"] = new_row["Profit"] - df.loc[date - 1]["Profit"]

    new_row["Profit d2P/dD2"] = new_row["Profit dP/dD"] - df.loc[date - 1]["Profit dP/dD"]

    new_row["Profit d3P/dD3"] = new_row["Profit d2P/dD2"] - df.loc[date - 1]["Profit d2P/dD2"]

    print("new row profit done", new_row)

  new_row_typed = {}

  for k, v in new_row.items():

    # print(k, v)

    new_row_typed[k] = float(v)

  print("new_row_typed", new_row_typed)

  return new_row_typed

<a name="your-leader"></a>
# Your Leader

Observations about black-box code:

*  Only classes that inherit from Leader will show in GUI (cannot make another intermediate base class)



In [14]:
group_num = 23

assert isinstance(group_num, int), f"Expected an integer for group_num, but got {type(group_num).__name__}"

In [15]:
class Optimum_chaser_generic(Leader):

  ##  Note that this is also a base class;
  ##  you should only instantiate instances of the MK* subclasses.

  def __init__(self, name, engine):

    ##  Load data, setup DFs.

    dfs_loaded = pd.read_excel('data.xlsx', sheet_name = None, index_col = "Date")

    self.dfs_primed = loaded_dfs_setup(dfs_loaded)

    ##  Setup new DF for new 30 days.

    columns_set = list(self.dfs_primed[self.target])

    self.df_30 = pd.DataFrame(columns = columns_set)

    self.df_orientation = pd.DataFrame(columns = ["Leader's Price",
                                                  "Profit",
                                                  "Big dP/dL",
                                                  "Tiny dP/dL"])

    self.df_grads = pd.DataFrame(columns = [
        "Leader's Price",
        "dP/dL",
        "Bound"
        ])

    self.exploit_initialised = 0


  def new_price(self, date: int):

    print("new_price()")

    if (date == 101):

      self.df_30.loc[100] = self.stats_hist

    else:

      print(self.df_30)

      print("date - 1: ", date - 1)

      print("cost: ", self.dfs_primed[self.target].iloc[-1]["Cost"])

      print("prices: ", self.engine.exposed_get_price(date - 1))

      new_row = update_history(
          self.df_30,
          date - 1,
          self.dfs_primed[self.target].iloc[-1]["Cost"],
          self.engine.exposed_get_price(date - 1),
          self.stats_hist
          )

      print("generated new_row: ", new_row)

      self.df_30.loc[date - 1] = new_row

      print(self.df_30)

      print(self.df_30.to_csv())


class Optimum_chaser_MK1(Optimum_chaser_generic, Leader):

    def __init__(self, name, engine):

        self.target = "Follower_Mk1"

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)


class Optimum_chaser_MK2(Optimum_chaser_generic, Leader):

    def __init__(self, name, engine):

        self.target = "Follower_Mk2"

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)


class Optimum_chaser_MK3(Optimum_chaser_generic, Leader):

    def __init__(self, name, engine):

        self.target = "Follower_Mk3"

        Optimum_chaser_generic.__init__(self, name, engine)
        Leader.__init__(self, name, engine)

        self.explore_exploit_boundary = 116

        ##  Set initial dL/dD value; this is a hyperparameter.

        initial_boost = 5

        self.stats_hist = extract_initial_stats(
            self.dfs_primed[self.target],
            initial_boost
            )

    def new_price(self, date: int):

        print("new_price() MK3")

        Optimum_chaser_generic.new_price(self, date)

        if (date == 101):

          return self.stats_hist["Leader's Price"] + self.stats_hist["Leader's Price dL/dD"]

        elif (date > 101 and date < self.explore_exploit_boundary):

          self.df_30, new_date_boundary, lp = explore_prices(
              self.df_30,
              date
              )

          if (new_date_boundary != 0):

            self.explore_exploit_boundary = new_date_boundary

          return lp

        else:

          if date <= self.explore_exploit_boundary and self.exploit_initialised == 0:

            self.exploit_initialised = 1

            lp = exploit_initialise_part_1(
                self.df_30
                )

            return lp

          elif self.exploit_initialised == 1:

            self.exploit_initialised = 2

            self.df_orientation = exploit_initialise_part_2(
                self.df_30,
                self.df_orientation,
                date,
                self.explore_exploit_boundary,
                self.engine.exposed_get_price(date - 1)
            )

          else:

            self.df_grads, lp = exploit_prices_gradient_gen(
                self.df_30,
                self.df_orientation,
                self.df_grads,
                date,
                self.explore_exploit_boundary,
                self.engine.exposed_get_price(date - 1)
            )

In [16]:
def exploit_prices_gradient_gen(df_30: pd.DataFrame,
                                df_grads: pd.DataFrame,
                                date: int,
                                date_boundary: int,
                                ##  dl_dp_hyperparam: float,
                                prices: tuple[int, int]) -> tuple[pd.DataFrame, float]:

  ##  Exploit; seek optimum of best known maximum on curve.
  ##  Find most profitable leader's price known so far.

  ##  This function takes the previous gradient, and generates a new gradient.

  ##  1. Finds most profitable LP value.

  new_df_row = {}

  if df_grads.empty:

    ##  2.a. Setup bounds for binary search.

    ##  Take the point when the gradient is 0, and the point
    ##  when the gradient is half the next point in df_30.

    most_profitable = df_30.loc[:date_boundary].loc[df_30["Profit"].idxmax()]

    self.df_grads.loc[0] = {
        "Leader's Price": most_profitable["Leader's Price"],
        "dP/dL": 0,
        "Bound": ["L"]
    }

    new_df_row = {
        "Leader's Price": most_profitable["Leader's Price"],
        "dP/dL": most_profitable["Leader's Price dL/dD"] / 2,
        "Bound": ["R"]
    }

    new_lp = most_profitable["Leader's Price"] + most_profitable["Leader's Price dL/dD"] / 2

  else:

    ##  2.b. Post-initialisation.

    ##  Run a binary search to generate new gradient, based on past gradients generated.

    last_entry = df_grads.iloc[-1]["Leader's Price dL/dD"]

    df_grads = update_bounds(df_grads)

    new_df_row = {
        "Leader's Price": df_grads.head(1)["Leader's Price"],
        "dP/dL": gen_midpoint_grad(df_grads),
        "Bound": []
    }

  df_grads.loc[date - 1] = new_df_row

  return df_grads, new_df_row["Leader's Price"] + new_df_row["dP/dL"]

In [17]:
def gen_midpoint_grad(df_grads: pd.DataFrame):

  l_bound = df_grads[df_grads["Bound"] == ["L"]]

  r_bound = df_grads[df_grads["Bound"] == ["R"]]

  return df_grads.loc[l_bound]["dL/dD"] + df_grads.loc[l_bound]["dL/dD"] / 2

In [18]:
def update_bounds(df_grads: pd.DataFrame):

  ##  Take the last row in dataframe, extract gradient.

  last_grad_added = df_grads.iloc[-1]["Leader's Price dL/dD"]

  ##  Duplicate dataframe.

  df_grads_addendum = df_grads.copy()

  ##  Find absolute difference between extracted gradient and all others.

  df_grads_addendum["dL/dD diff abs"] = (
      df_grads["Leader's Price dL/dD"] - last_grad_added
  ).abs()

  ##  Sort by gradient.

  df_grads_diffs = df_grads_addendum.sort_values(by = "dL/dD diff abs")

  boundaries = {
      "L":
      df_grads_diffs[
      df_grads_diffs["dL/dD"] < last_grad_added
      ].iloc[0],
      "R":
      df_grads_diffs[
      df_grads_diffs["dL/dD"] > last_grad_added
      ].iloc[0]
  }

  ##  Wipe previously stored boundaries.

  df_grads["Bound"] = []

  ##  Find closest boundary.

  # boundary_dates = {
  #     "L":
  #     df_grads[df_grads_diffs[""] == boundaries["L"]["dL/dD diff abs"]]iloc[0]
  # }

  # boundaries["L"]["dL/dD"]

  ##  Compare distances to both boundaries. Choose closest two.

  if boundaries["L"]["dL/dD diff abs"] > boundaries["R"]["dL/dD diff abs"]:

    ##  Closest is R and centre.

    new_l_loc = df_grads.index[-1]

    new_r_loc = df_grads[df_grads["dL/dD"] == boundaries["R"]["dL/dD"]]

  elif boundaries["L"]["dL/dD diff abs"] < boundaries["R"]["dL/dD diff abs"]:

    ##  Closest is L and centre.

    new_l_loc = df_grads[df_grads["dL/dD"] == boundaries["L"]["dL/dD"]]

    new_r_loc = df_grads.index[-1]

  df_grads.loc[new_l_loc]["Bound"] = ["L"]

  df_grads.loc[new_r_loc]["Bound"] = ["R"]

In [19]:
def exploit_initialise_part_2(df_30: pd.DataFrame,
                              df_orientation: pd.DataFrame,
                              date: int) -> tuple[pd.DataFrame, float]:

  ##  Exploit; seek optimum of best known maximum on curve.
  ##  Find most profitable leader's price known so far.

  ##  This function finds dP/dL based on the tiny increment in price, and stores it.

  ##  The dataframe is expected to contain only 1 row, consisting solely of constants.

  new_df_row = {}

  ##  Take 2 rows with largest profit value (should contain LP and LP + 0.1).
  ##  Ensure sorted by date (index column).

  profitable_top_two = df_30.nlargest(2, "Profit").sort_index()

  new_df_row["Leader's Price"] = profitable_top_two["Leader's Price"].iloc[0]

  new_df_row["Profit"] = profitable_top_two["Profit"].iloc[0]

  new_df_row["Big dP/dL"] = profitable_top_two["dP/dL"].iloc[0]

  ##  Calculate differences between profit values, select lowest row.

  new_df_row["Tiny dP/dL"] = profitable_top_two["Profit"].diff().fillna(0).iloc[-1]

  df_orientation.loc[date - 1] = new_df_row

  return df_orientation

In [20]:
def exploit_initialise_part_1(df_30: pd.DataFrame) -> tuple[pd.DataFrame, float]:

  ##  Exploit initialisation.

  ##  This function finds the most profitable LP value, and adds a small increment.

  most_profitable = df_30.loc[df["Profit"].idxmax()]

  date = df["Profit"].idxmax()

  lp = most_profitable["Leader's Price"]

  lp_inc = lp + 0.1

  return lp_inc

In [21]:
def explore_prices(df: pd.DataFrame,
                   date: int) -> tuple[pd.DataFrame, float]:

  print(f"explore_prices(), {date}")

  ##  Explore; seek maximums along fluctuating curve.
  ##  Note that this sequence constantly grows leader's price.

  if (date <= 103):

    return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] * 2)

  elif df.loc[date - 1]["Profit"] < 0:

    ##  Escape hatch here in case profitability becomes negative.

    ##  Date boundary will need updating.

    return df, date - 1,  df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] * 2)

  else:

    ##  If 2nd derivative of profitability is higher than previously, and positive (curve on steepening incline)...

    if (df.loc[date - 1]["Profit d2P/dD2"] > df.loc[date - 2]["Profit d2P/dD2"] and df.loc[date - 1]["Profit d2P/dD2"] > 0):

      ##  Double rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] * 2)

    elif df.loc[date - 1]["Profit d2P/dD2"] < df.loc[date - 2]["Profit d2P/dD2"] and df.loc[date - 1]["Profit d2P/dD2"] > 0:

      ##  2nd derivative is decreasing (curve on flattening incline, slow down whilst passing optimum)...

      ##  Half rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] / 2)

    else:

      ##  2nd derivative is negative (curve on decline)...

      ##  Double rate of change of leader's price...

      return df, 0, df.loc[date - 1]["Leader's Price"] + (df.loc[date - 1]["Leader's Price dL/dD"] + df.loc[date - 1]["Leader's Price d2L/dD2"] * 2)


Please implement your leaders below. Please clearly document:
*   State which leader(s) is designed to play against each follower.

# Simulation

Below is the GUI interface. Please select a leader from the dropdown menu and a follower from the dropdown menu, then click “Connect.” Once the status updates to “Connected to *your_selected_leader* and *your_selected_follower*” click “Start Simulation” to begin. If you wish to save the generated data, click “Export Data.” The dataset will be saved in the “run” folder.

In [22]:
display(Javascript('''google.colab.output.setIframeHeight(0, true, {maxHeight: 5000})'''))
engine = Engine()
app = GUI(engine, Leader, group_num)

<IPython.core.display.Javascript object>

1.8071091994472521


In [23]:
leader = Optimum_chaser_MK3("test_mk1", engine)

print(leader.stats_hist)

print(leader.new_price(101))

print(leader.df_30)

new_row = update_history(
    leader.df_30,
    101,
    leader.dfs_primed["Follower_Mk1"].iloc[-1]["Cost"],
    (leader.new_price(101), 20),
    leader.stats_hist
    )

filtered_dict = {k: float(v) for k, v in new_row.items() if k in leader.df_30.columns}

print(filtered_dict)

leader.df_30.loc[101] = new_row

#print(leader.df_30)

print(leader.df_30.head(2))

1.8071091994472521
{"Leader's Price": 1.0, "Follower's Price": 1.805600345606945, 'Cost': 1.0, "Leader's Price dL/dD": 0.09521615689527474, "Leader's Price d2L/dD2": 0.0, 'Profit': 0.0, 'Profit dP/dD': 0.0, 'Profit d2P/dD2': 0.0, 'Profit d3P/dD3': 0.0}
new_price() MK3
new_price()
1.0952161568952747
     Leader's Price  Follower's Price     Cost  Leader's Price dL/dD  \
100        1.000000          1.805600 1.000000              0.095216   

     Leader's Price d2L/dD2   Profit  Profit dP/dD  Profit d2P/dD2  \
100                0.000000 0.000000      0.000000        0.000000   

     Profit d3P/dD3  
100        0.000000  
new_price() MK3
new_price()
update_history()
1.0 1.0952161568952747 20
new row in progress for date 101 {'Cost': np.float64(1.0), "Leader's Price": 1.0952161568952747, "Follower's Price": 20, 'Profit': np.float64(14.713173736098057)}
new_row_typed {'Cost': 1.0, "Leader's Price": 1.0952161568952747, "Follower's Price": 20.0, 'Profit': 14.713173736098057, "Leader's Pric

**Initial data exploration**

In [24]:
# from IPython.display import SVG, display

# # Replace with your Raw GitHub URL
# url = "https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg"

# display(SVG(url=url))